In [ ]:
%pip install engineeringkit


# Bowman, Mueller and Nagle: the LMTD correction factor (1940)

The log-mean temperature difference is exact only for pure counterflow or
parallel flow. In a shell-and-tube exchanger the tube fluid passes back and
forth while the shell fluid crosses it, and in a crossflow core neither
stream sees a single temperature. R. A. Bowman, A. C. Mueller and W. M.
Nagle, collecting the analyses of the 1930s, presented the correction as
one factor $F$ on the counterflow LMTD, charted against the
effectiveness $P$ of one stream and the capacity ratio $R$
(:func:`~engineeringkit.heat_transfer.lmtd_correction_factor`). The
effectiveness-NTU forms of the same arrangements
(:func:`~engineeringkit.heat_transfer.exchanger_effectiveness_shell_tube`,
:func:`~engineeringkit.heat_transfer.exchanger_effectiveness_crossflow`)
carry the same information. This example redraws the chart, checks the two
methods against each other and rates a two-shell oil cooler.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from engineeringkit.heat_transfer import (
    exchanger_effectiveness_counterflow,
    exchanger_effectiveness_crossflow,
    exchanger_effectiveness_parallel,
    exchanger_effectiveness_shell_tube,
    lmtd_correction_factor,
    log_mean_temperature_difference,
)

## The F chart for one shell pass

Each curve ends where $F$ plunges to zero: the outlet temperatures
would have to cross, which one shell pass cannot do. Designers stay above
$F \approx 0.75$.



In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for R in [0.2, 0.6, 1.0, 1.5, 3.0]:
    P = np.linspace(0.0, 0.99, 400)
    F = []
    for p in P:
        try:
            F.append(lmtd_correction_factor(p, R))
        except ValueError:
            F.append(np.nan)
    ax.plot(P, F, label=f"R = {R:g}")
ax.axhline(0.75, color="gray", ls=":", lw=0.8)
ax.set(xlabel="P = (t_o - t_i)/(T_i - t_i)", ylabel="correction factor F", ylim=(0.5, 1.01), title="One shell pass, 2, 4, ... tube passes")
ax.legend()
fig.tight_layout()

## Two methods, one answer

From the effectiveness, the outlet temperatures and therefore $P$,
the LMTD and $F$ all follow; $UAF\Delta T_{lm}$ must return the
same duty. With the tube stream the smaller one, $P = \varepsilon$
and $R = C_r$.



In [ ]:
worst = 0.0
for shells in [1, 2, 4]:
    for NTU in [0.5, 1.0, 2.0]:
        for Cr in [0.25, 0.75, 1.0]:
            P = float(exchanger_effectiveness_shell_tube(NTU, Cr, shells))
            duty = NTU * lmtd_correction_factor(P, Cr, shells) * log_mean_temperature_difference(1 - P, 1 - Cr * P)
            worst = max(worst, abs(duty / P - 1))
print(f"largest disagreement between F-LMTD and effectiveness-NTU: {worst:.1e}")
assert worst < 1e-10

## Where the arrangements sit

Every arrangement lies between parallel flow and counterflow. Adding shells
approaches counterflow; crossflow with both streams unmixed sits close to
one shell pass.



In [ ]:
NTU, Cr = np.linspace(0, 5, 100), 0.75
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(NTU, exchanger_effectiveness_counterflow(NTU, Cr), label="counterflow")
for shells in [1, 3]:
    ax.plot(NTU, exchanger_effectiveness_shell_tube(NTU, Cr, shells), label=f"shell-and-tube, {shells} shell(s)")
ax.plot(NTU, exchanger_effectiveness_crossflow(NTU, Cr), "--", label="crossflow, both unmixed")
ax.plot(NTU, exchanger_effectiveness_parallel(NTU, Cr), label="parallel flow")
ax.set(xlabel="NTU", ylabel="effectiveness", title=f"Cr = {Cr}")
ax.legend()
fig.tight_layout()

## Rating a two-shell oil cooler

Oil ($C = 2000$ W/K) enters the shell at 140 °C and water
($C = 2500$ W/K) the tubes at 20 °C; $UA = 3000$ W/K. With two
shell passes, what are the outlets, and how much more area would one shell
pass need for the same duty?



In [ ]:
C_oil, C_water, UA, T_in, t_in = 2000.0, 2500.0, 3000.0, 140.0, 20.0
C_min, Cr = C_oil, C_oil / C_water
eps = float(exchanger_effectiveness_shell_tube(UA / C_min, Cr, shells=2))
q = eps * C_min * (T_in - t_in)
T_out, t_out = T_in - q / C_oil, t_in + q / C_water
P, R = (t_out - t_in) / (T_in - t_in), C_water / C_oil
F2 = lmtd_correction_factor(P, R, shells=2)
dT_lm = log_mean_temperature_difference(T_in - t_out, T_out - t_in)
print(f"two shells: ε = {eps:.3f}, q = {q / 1e3:.1f} kW, oil out {T_out:.1f} °C, water out {t_out:.1f} °C, F = {F2:.3f}")
assert np.isclose(UA * F2 * dT_lm, q, rtol=1e-10)
F1 = lmtd_correction_factor(P, R, shells=1)
UA_one = q / (F1 * dT_lm)
print(f"one shell pass would need F = {F1:.3f}, so UA = {UA_one:.0f} W/K ({UA_one / UA - 1:.0%} more area)")
assert F1 < F2 and UA_one > UA

plt.show()